In [61]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.integrate import ode

- Funkcije

In [64]:
def integratorLorenz63(
        X0, Y0, Z0, integration_time,
        sigma=10., b = 8./3, r=28.
        ):
    """
    Integracija Lorenzovega '63 modela
    Args:
    - X0, Y0, Z0 = X(t=0), Y(t=0), Z(t=0) -> začetni pogoji
    - integration_time -> 1 enota = 100 korakov pri dt=0.01
    - sigma, b, r -> konstante sistema enačb
    """

    # Funkcija za integracijo: parcialno X/parcialno tau
    def f(t, X):
        # Desna stran Lorenz '63 sistema
        return [sigma*(X[1]-X[0]),
                X[0]*(r-X[2])-X[1],
                X[0]*X[1]-b*X[2]]
    
    def jac(t, X):
    # Jakobian sistema
        return [[-sigma, sigma, 0],
                [r-X[2], X*(r-X[2])-1, -X[0]],
                [X[1], X[0], -b]]
    
    # Integrator Runge-Kutta reda (4)5
    Integrator = ode(f, jac).set_integrator('dopri5')

    # Začetni pogoj za integracijo
    X0 = [X0, Y0, Z0] # Pozor - tu redefiniramo X0
    t0 = 0
    Integrator.set_initial_value(X0, t0)

    # Časovni korak integracije
    dt = 0.01

    # Tabeli za shranjevanje rezultatov integracije
    t = [0.]
    X = [X0]

    # Integracija
    while Integrator.successful() and Integrator.t+dt < integration_time:
        t.append(float(Integrator.t+dt))
        X.append(list(Integrator.integrate(Integrator.t+dt)))
        
    return np.array(t), np.array(X)


# Sekvenciranje multivariatnega zaporedja
def split_into_sequences(
        multivariate_sequence, input_seq_len,
        start_every_num_steps=8):
    """
    multivariate_sequence: shape (T, d); T=# čas. korakov, d=# dimenzij
    input_seq_len: dolžina vhodnega zaporedja
    start_every_num_steps: koliko časovnih korakov izpustimo,
    preden shranimo novo sekvenco
    """

    X, y = list(), list()
    
    # Zanka skozi vse časovne korake
    for i in range(0, len(multivariate_sequence), start_every_num_steps):

        # Zadnji indeks sekvence
        end_ix = i + input_seq_len

        # Preveri če je zadnji indeks izven multivariate_sequence
        if end_ix > len(multivariate_sequence)-1:
            break

        # seq_x je sestavljena iz input_seq_len korakov
        seq_x = multivariate_sequence[i:end_ix, :]
        # seq_y je samo 1 element za napoved
        seq_y = multivariate_sequence[end_ix, :]

        X.append(seq_x)
        y.append(seq_y)

    return np.array(X), np.array(y)


# Custom layer for Luong Attention
class LuongAttention(tf.keras.layers.Layer):
    def __init__(self):
        super(LuongAttention, self).__init__()
    
    def call(self, decoder_hidden, encoder_outputs):
        # Izračun skalarnega produkta med skritim stanjem dekodirnika
        # in skritimi stanji kodirnika. V ta namen za vsak primer v batchu
        # množimo matriko z vrsticami skritih stanj kodirnika s
        # transponirano matriko skritih stanj dekodirnika (tf.matmul).
        score = tf.matmul(encoder_outputs, decoder_hidden, transpose_b=True)
        # shape = (batch_size, time_steps_encoder, time_steps_decoder=1)

        # Izračunamo uteži pozornosti - softmax po osi kodirnih skritih stanj
        attention_weights = tf.nn.softmax(score, axis=1)
        # shape = (batch_size, time_steps_encoder, time_steps_decoder=1)

        # Vektor konteksta je s pozornostjo utežena linearna kombinacija
        # izhodov kodirnika.
        context_vector = tf.reduce_sum(
            attention_weights * encoder_outputs, axis=1)
        # shape = (batch_size, hidden_size)

        return context_vector, attention_weights

In [13]:
# Za ponovljivost določimo seme
np.random.seed(42)

#Začetni pogoj za učno trajektorijo
X0_train = float(np.random.uniform(low=-50., high = 50., size = 1))
Y0_train = float(np.random.uniform(low=-50., high = 50., size = 1))
Z0_train = float(np.random.uniform(low=-50., high = 50., size = 1))

# Čas integracije
t_train = 8030. # Prvih 30 enot časa bomo odstranili

# Numerična integracija
t_train, data_train = integratorLorenz63(
    X0_train, Y0_train, Z0_train,
    t_train)

# Odrežemo prvih 30 enot časa - da ohranimo le točke na atraktorju
relaksacijski_indeks = 3000
data_train = np.array(data_train)[relaksacijski_indeks:, :]

# Podatke razdelimo na učno in validacijsko množico
# 90 % za učenje, 10 % za validacijo
split_idx = int(0.1 * len(data_train))

X_val = data_train[:split_idx]
X_train = data_train[split_idx:]

C:\Users\tadej\AppData\Local\Temp\ipykernel_27792\883031244.py:5: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  X0_train = float(np.random.uniform(low=-50., high = 50., size = 1))
C:\Users\tadej\AppData\Local\Temp\ipykernel_27792\883031244.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Y0_train = float(np.random.uniform(low=-50., high = 50., size = 1))
C:\Users\tadej\AppData\Local\Temp\ipykernel_27792\883031244.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)

In [16]:
print(X_train.shape)
print(X_val.shape)

(720000, 3)
(80000, 3)


In [20]:
# Uporabimo standardizacijo (opcija je tudi MinMaxScaler)
scaler = StandardScaler()
# Povprečje in std izračunamo na učni množici (ne na validacijski)!
scaler.fit(X_train)

# Izpiši povprečno vrednost in standardno deviacijo
print("Povprečje:", scaler.mean_)
print("Standardna deviacija:", scaler.scale_)

# Normiramo podatke
X_train_norm = scaler.transform(X_train)
X_val_norm = scaler.transform(X_val)

Povprečje: [-0.03917021 -0.0391126  23.55120557]
Standardna deviacija: [7.92470512 9.01091533 8.62086367]


In [22]:
input_seq_len = 5
# Učni podatki
X1_train, Y1_train = split_into_sequences(X_train_norm, input_seq_len)
# Validacijski podatki
X1_val, Y1_val = split_into_sequences(X_val_norm, input_seq_len)

In [ ]:
model = tf.keras.models.Sequential()
model.add(SimpleRNN(
    units=16, # dimenzija skritega stanja
    return_sequences=False,
    input_shape=(None,3))
    )
# ekvivalent za LSTM: model.add(LSTM(...))
# Zadnje skrito stanje še preslikamo v izhodni vektor y
model.add(Dense(3))
# Nastavitve za učenje modela
model.compile(loss='mean_squared_error', optimizer='Adam')

In [ ]:
"""   KODIRNIK   """
# Vhodni tenzor za kodirnik
encoder_inputs = tf.keras.Input(shape=(None, 3))
# Kodirni sloj
encoder_lstm = layers.LSTM(
    units=16, # Število dimenzij skritega stanja
    return_sequences=True, # Vrni vsa skrita stanja
    return_state=True, # Vrni zadnje celično stanje
    name='encoder' # Sloj poimenujemo za kasnejšo rabo
)

# Pošlji vhodni tenzor kodirnika skozi kodirnik
encoder_hidden_states, state_h, state_c = encoder_lstm(encoder_inputs)
# encoder_hidden_states shape: # (batch_size, encoder_time_steps, hidden_size)


"""   DEKODIRNIK   """
# Vhodni tenzor za dekodirnik
decoder_inputs = tf.keras.Input(shape=(None, 3))
# Dekodirni sloj
decoder_lstm = layers.LSTM(
    units=16, # Število dimenzij skritega stanja
    return_sequences=True, # Vrni vsa skrita stanja
    return_state=True, # Vrni zadnje celično stanje
    name='decoder' # Sloj poimenujemo za kasnejšo rabo
)

# Nastavi začetno skrito in celično stanje dekodirnika
# in pošlji vhodni tenzor dekodirnika skozi dekodirnik
decoder_hidden_states, _, _ = decoder_lstm(
    decoder_inputs, initial_state=[state_h, state_c])
# decoder_hidden_states shape: # (batch_size, decoder_time_steps, hidden_size)


"""   Luongov mehanizem pozornosti (Luong Attention)   """
attention_layer = LuongAttention()
context_vector, attention_weights = attention_layer(
    decoder_hidden_states, encoder_hidden_states)

In [ ]:
# Konkateniramo vektorje konteksta s skritimi stanji dekoderja
combined_context = layers.Concatenate(axis=-1)(
[decoder_hidden_states, tf.expand_dims(context_vector, 1)])

# Konkatenirane vektorje preslikamo v nova skrita stanja, tako da
# jih pošljemo skozi gosto povezano NN. V ta namen uporabimo sloj
# TimeDistributed, ki aplicira uteži na vse izhode.
new_decoder_hidden_states = layers.TimeDistributed(
    layers.Dense(units=16, activation="tanh"))(combined_context)
# Nova skrita stanja z gosto povezano NN preslikamo v izhode.

outputs = layers.TimeDistributed(
    layers.Dense(3))(new_decoder_hidden_states)
# Ustvarimo model (funkcijo oz. računski graf), tako da definiramo

# vhodna in izhodna polja.
model = tf.keras.Model(
    inputs=[encoder_inputs, decoder_inputs],
    outputs = outputs)

# Konfiguriramo nastavitve za učenje
model.compile(loss='mean_squared_error', optimizer='Adam')